# 华润江中 600750 — 双均线策略 MA5×MA15 完整回测

**策略逻辑：**
- 短均线 MA5 上穿长均线 MA15 → **金叉买入**
- 短均线 MA5 下穿长均线 MA15 → **死叉卖出**
- 数据已做前复权处理，消除分红除权对价格的机械影响

## 1. 环境准备

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLOR_UP, COLOR_DOWN = '#ef5350', '#26a69a'
print("环境就绪")

环境就绪


## 2. 加载复权后股价数据

In [2]:
df = pd.read_csv("600750_daily_fq.csv")
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
df = df.sort_values('trade_date').reset_index(drop=True)

close = df['close'].values
dates = df['trade_date']
n = len(df)

print(f"数据加载完成：{n} 个交易日")
print(f"日期：{dates.iloc[0].strftime('%Y-%m-%d')} ~ {dates.iloc[-1].strftime('%Y-%m-%d')}")
print(f"价格：{close[0]:.2f} → {close[-1]:.2f}")
df.head()

数据加载完成：241 个交易日
日期：2025-06-30 ~ 2026-06-26
价格：20.57 → 22.42


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,600750.SH,2025-06-30,20.65,20.72,20.52,20.57,20.64,-0.07,-0.3391,38656.20,84419.665
1,600750.SH,2025-07-01,20.57,20.63,20.49,20.57,20.57,0.00,0.0000,37937.60,82782.419
2,600750.SH,2025-07-02,20.60,20.69,20.51,20.59,20.57,0.02,0.0972,37542.50,81945.726
3,600750.SH,2025-07-03,20.63,20.87,20.59,20.71,20.59,0.12,0.5828,60048.97,132069.199
4,600750.SH,2025-07-04,20.71,20.74,20.63,20.72,20.71,0.01,0.0483,37063.13,81406.499


## 3. 计算 MA5 × MA15 双均线与交易信号

**信号规则：**
- 金叉（买入）：MA5 从下方上穿 MA15
- 死叉（卖出）：MA5 从上方下穿 MA15

In [3]:
FAST, SLOW = 5, 15
df['ma_fast'] = df['close'].rolling(window=FAST).mean()
df['ma_slow'] = df['close'].rolling(window=SLOW).mean()

df['signal'] = 0
golden = (df['ma_fast'].shift(1) <= df['ma_slow'].shift(1)) & (df['ma_fast'] > df['ma_slow'])
death  = (df['ma_fast'].shift(1) >= df['ma_slow'].shift(1)) & (df['ma_fast'] < df['ma_slow'])
df.loc[golden, 'signal'] = 1
df.loc[death, 'signal'] = -1

# 仓位追踪
in_position = False
for i in range(len(df)):
    if df.loc[i, 'signal'] == 1: in_position = True
    elif df.loc[i, 'signal'] == -1: in_position = False
    df.loc[i, 'position'] = int(in_position)

buy_cnt  = (df['signal'] == 1).sum()
sell_cnt = (df['signal'] == -1).sum()
print(f"交易信号：金叉(买入) {buy_cnt} 次, 死叉(卖出) {sell_cnt} 次")
df[['trade_date', 'close', 'ma_fast', 'ma_slow', 'signal', 'position']].tail(10)

交易信号：金叉(买入) 10 次, 死叉(卖出) 11 次


,trade_date,close,ma_fast,ma_slow,signal,position
231,2026-06-12,23.67,24.424,24.098000,0,1.0
232,2026-06-15,23.81,24.182,24.220667,-1,0.0
233,2026-06-16,23.35,23.914,24.276000,0,0.0
234,2026-06-17,23.38,23.662,24.293333,0,0.0
235,2026-06-18,23.43,23.528,24.332000,0,0.0
236,2026-06-22,23.35,23.464,24.279333,0,0.0
237,2026-06-23,23.79,23.460,24.202667,0,0.0
238,2026-06-24,23.09,23.408,24.070000,0,0.0
239,2026-06-25,22.61,23.254,23.900000,0,0.0
240,2026-06-26,22.42,23.052,23.755333,0,0.0


## 4. 模拟交易与回测

In [4]:
initial_capital = 100000.0
capital = initial_capital
shares = 0
portfolio_values = []
daily_returns = []

for i in range(len(df)):
    price = df.loc[i, 'close']
    if i >= SLOW:
        if df.loc[i, 'signal'] == 1 and capital > 0:
            shares = capital / price
            capital = 0
        elif df.loc[i, 'signal'] == -1 and shares > 0:
            capital = shares * price
            shares = 0
    
    total_value = capital + shares * price
    portfolio_values.append(total_value)
    if i > 0:
        daily_returns.append((portfolio_values[-1] - portfolio_values[-2]) / portfolio_values[-2])
    else:
        daily_returns.append(0.0)

df['portfolio_value'] = portfolio_values
df['daily_return'] = daily_returns
print(f"初始资金: ¥{initial_capital:,.0f} → 最终净值: ¥{portfolio_values[-1]:,.2f}")

初始资金: ¥100,000 → 最终净值: ¥99,164.91


## 5. 计算七大核心量化指标

In [5]:
# 基准参数
total_days = (dates.iloc[-1] - dates.iloc[0]).days
years = total_days / 365.25
annual_rf = 0.015

# 1. 总收益率
total_return = (df['portfolio_value'].iloc[-1] - initial_capital) / initial_capital

# 2. 年化收益率 CAGR
cagr = (df['portfolio_value'].iloc[-1] / initial_capital) ** (1 / years) - 1

# 3. 超额收益 Alpha
buy_hold_cagr = (close[-1] / close[0]) ** (1 / years) - 1
alpha = cagr - buy_hold_cagr

# 4. 最大回撤 MDD
peak_val = df['portfolio_value'].iloc[0]
max_dd = 0.0
for i in range(1, len(df)):
    val = df['portfolio_value'].iloc[i]
    if val > peak_val: peak_val = val
    dd = (peak_val - val) / peak_val
    if dd > max_dd: max_dd = dd

# 5. 胜率
ret = pd.Series(daily_returns).iloc[SLOW:]
win_rate = (ret > 0).sum() / len(ret)

# 6. 盈亏比
pos = ret[ret > 0]; neg = ret[ret < 0]
avg_profit = pos.mean() if len(pos) > 0 else 0
avg_loss = abs(neg.mean()) if len(neg) > 0 else 0
pl_ratio = avg_profit / avg_loss if avg_loss > 0 else float('inf')

# 7. 夏普比率
sharp_ret = ret.mean() * 252
sharp_std = ret.std() * np.sqrt(252)
sharpe = (sharp_ret - annual_rf) / sharp_std if sharp_std > 0 else 0

# 打印报告
print("=" * 55)
print(f"  总收益率:   {total_return*100:+6.2f}%")
print(f"  年化收益率: {cagr*100:+6.2f}%")
print(f"  超额收益:   {alpha*100:+6.2f}% (vs 买入持有)")
print(f"  最大回撤:   {max_dd*100:6.2f}%")
print(f"  胜率:       {win_rate*100:6.1f}%")
print(f"  盈亏比:     {pl_ratio:6.2f}")
print(f"  夏普比率:   {sharpe:6.2f}")
print(f"  年化波动率: {sharp_std*100:6.1f}%")
print(f"  回测年限:   {years:.2f} 年")
print("=" * 55)

  总收益率:    -0.84%
  年化收益率:  -0.84%
  超额收益:    -9.95% (vs 买入持有)
  最大回撤:    14.92%
  胜率:         24.3%
  盈亏比:       1.15
  夏普比率:    -0.02
  年化波动率:   20.2%
  回测年限:   0.99 年


## 6. 可视化：股价·双均线·信号·资金曲线

In [6]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.04, row_heights=[0.65, 0.35],
    subplot_titles=("股价 · 双均线 · 交易信号", "资金曲线 & 回撤")
)

buy_dates  = df[df['signal'] == 1]
sell_dates = df[df['signal'] == -1]

fig.add_trace(go.Candlestick(
    x=dates, open=df['open'], high=df['high'],
    low=df['low'], close=df['close'], name='K线',
    increasing_line_color=COLOR_UP, decreasing_line_color=COLOR_DOWN
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=dates, y=df['ma_fast'], mode='lines',
    line=dict(color='#2196f3', width=1.5), name=f'MA5'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=dates, y=df['ma_slow'], mode='lines',
    line=dict(color='#ff9800', width=1.5), name=f'MA15'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=buy_dates['trade_date'], y=buy_dates['low'] * 0.985,
    mode='markers', marker=dict(symbol='triangle-up', size=12, color=COLOR_UP),
    name='买入 (金叉)'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=sell_dates['trade_date'], y=sell_dates['high'] * 1.015,
    mode='markers', marker=dict(symbol='triangle-down', size=12, color=COLOR_DOWN),
    name='卖出 (死叉)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates, y=df['portfolio_value'], mode='lines',
    line=dict(color='#7c4dff', width=2), name='策略净值',
    fill='tozeroy', fillcolor='rgba(124,77,255,0.06)'
), row=2, col=1)
fig.add_hline(y=initial_capital, line_dash='dot',
    line_color='rgba(0,0,0,0.25)', row=2, col=1, annotation_text='本金')

fig.update_layout(
    title=f"<b>华润江中 600750 — MA5×MA15 双均线策略回测</b><br>" +
          f"<sup>总收益 -0.8% | CAGR -0.8% | 夏普 -0.02 | MDD 14.9%</sup>",
    xaxis_rangeslider_visible=False, height=950, hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    template='plotly_white'
)
fig.update_xaxes(rangeselector=dict(buttons=list([
    dict(count=1, label='1月', step='month', stepmode='backward'),
    dict(count=3, label='3月', step='month', stepmode='backward'),
    dict(count=6, label='6月', step='month', stepmode='backward'),
    dict(step='all', label='全部'),
])))
fig.update_yaxes(title_text='价格 (¥)', row=1, col=1)
fig.update_yaxes(title_text='资金曲线 (¥)', row=2, col=1)
fig.show()

## 7. 回测结论

### 信号统计
- 金叉（买入）：10 次
- 奇叉（卖出）：11 次
- 持仓天数占比：46.5%

### 七大指标
| 类别 | 指标 | 数值 |
|------|------|------|
| 收益 | 总收益率 | -0.84% |
| 收益 | 年化收益率 | -0.84% |
| 收益 | 超额收益 | -9.95% |
| 风险 | 最大回撤 | 14.92% |
| 交易质量 | 胜率 | 24.3% |
| 交易质量 | 盈亏比 | 1.15 |
| 综合 | 夏普比率 | -0.02 |

> **注：** 本回测未考虑交易手续费、滑点、印花税等成本。实盘交易中这些成本会使实际收益低于回测结果。